# HyDE를 이용한 Query Transformation

## 환경설정

In [2]:
from dotenv import load_dotenv
load_dotenv()

PINECONE_INDEX_NAME = 'adv-rag'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_METRIc = 'cosine'
PINECONE_INDEX_DEMENSION = 1536

OPENAI_LLM_MODEL = 'gpt-4.1-mini'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'

## 데이터 로드

In [3]:
import pandas as pd

document_df = pd.read_csv("data/documents.csv")
queries_df = pd.read_csv("data/queries.csv")
queries_df

,query_id,query_text,relevant_doc_ids
0,Q1,제주도 올레길 트레킹 코스 추천,D1=2
1,Q2,전주 비빔밥 vs 진주 비빔밥 재료 및 맛 차이,D2=3
2,Q3,걸스데이 대표 히트곡 목록 알려줘,D3=3
3,Q4,훈민정음 창제 배경과 세종대왕의 의의,D4=3
4,Q5,이순신 장군이 명량 해전에서 사용한 전술은 무엇인가?,D5=3
5,Q6,2024년 기후 변화 주요 지표와 한국의 탄소 중립 정책,D6=3
6,Q7,한국 AI 윤리 이슈와 관련 정책 사례는?,D7=3;D25=2
7,Q8,서울 지하철 환승 시 T-money 사용 방법,D8=2
8,Q9,판소리 춘향가 줄거리와 공연 특징,D9=3
9,Q10,한국 축구 대표팀 2002년 한일 월드컵 4강 진출 이유,D10=2


## 검색기 준비

In [4]:
from konlpy.tag import Okt
from rank_bm25 import BM25Okapi

okt = Okt()
tokenized_docs = [okt.morphs(content) for content in document_df['content']]
bm25 = BM25Okapi(tokenized_docs)

def bm25_search(query, top_k=5):
    """
    BM25로 질문과 관련 있는 상위 문서 ID를 반환한다.
    """
    query_token = okt.morphs(query)
    scores = bm25.get_scores(query_token)
    sorted_idx = sorted(range(len(scores)), key=lambda i:scores[i], reverse=True)
    ranked_docs = [document_df['doc_id'].iloc[i] for i in sorted_idx[:top_k]]
    return ranked_docs

## Dense 검색기 준비

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# 벡터스토어 생성
vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings
)

def dense_search(query,top_k=5):
    """Dense retrieval로 질문과 관련있는 상위 문서 ID를 반환한다."""
    docs = vector_store.similarity_search(query,k=top_k)
    return [doc.metadata["doc_id"] for doc in docs]

## Query Transform
: 사용자의 질문을 검색에 더 적합한 형태로 바꾸는 방법

## HyDE 기법
'HyDE(Hypothetical Document Embeddings)'는 사용자 질문에 대해 LLM으로 가상의 답변 또는 가상의 문서를 먼저 생성하고, 이 가상 문서를 임베딩하여 검색하는 방식이다.



In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from tqdm import tqdm

hype_prompt = PromptTemplate.from_template("""
아래의 질문에 대해 실제 문서가 아니어도 좋으니 검색에 도움이 되는 짧은 가상 답변을 생성하세요.
                                           
규칙:
- 질문의 핵심 키워드와 관련 개념을 포함하세요.
- 실제 문서처럼 3~4문장으로 작성하세요.
- 모르는 내용을 단정적으로 과장하지 말고, 검색에 필요한 표현을 중심으로 작성하세요.
                                           
질문 : {query}
""")

hyde_llm = ChatOpenAI(model=OPENAI_LLM_MODEL, temperature=0.3)
output_parser = StrOutputParser()

hyde_chain = hype_prompt | hyde_llm |output_parser

## 단일 질문으로 HyDE 흐름 확인

In [8]:
sample_query = queries_df.loc[0,'query_text']
sample_pseudo_answer = hyde_chain.invoke({'query':sample_query})

print("원본 질문:")
print(sample_query)
print("가상 답변")
print(sample_pseudo_answer)

원본 질문:
제주도 올레길 트레킹 코스 추천
가상 답변
제주도 올레길은 총 26개 코스로 구성되어 있으며, 각 코스마다 독특한 자연경관과 문화유산을 경험할 수 있습니다. 특히 7코스는 한라산과 해안을 동시에 즐길 수 있어 인기 있으며, 10코스는 성산 일출봉과 가까워 일출 명소로 유명합니다. 트레킹 난이도와 소요 시간, 주변 편의시설을 고려해 코스를 선택하는 것이 좋으며, 계절별 날씨 정보도 함께 확인하는 것을 추천합니다.


## 원본 질문 검색과 HyDE 검색 비교

In [9]:
sample_dense_ids = dense_search(sample_query,top_k=5)
sample_hyde_ids = dense_search(sample_pseudo_answer, top_k=5)

comparison_df = pd.DataFrame({
    'rank':range(1,6),
    'Dense':sample_dense_ids,
    'HyDE':sample_hyde_ids
})

comparison_df

,rank,Dense,HyDE
0,1,D1,D1
1,2,D12,D12
2,3,D8,D8
3,4,D2,D15
4,5,D23,D6


## 전체 질문에 대한 가상 답변 생성

In [10]:
hyde_pseudo = {}

for idx, row in tqdm(queries_df.iterrows(),total=len(queries_df)):
    qid = row['query_id']
    query = row['query_text']

    pseudo_answer = hyde_chain.invoke({'query':query})
    hyde_pseudo[qid] = pseudo_answer

100%|██████████| 30/30 [01:23<00:00,  2.77s/it]


In [11]:
pd.set_option('display.max_colwidth',None)

hyde_pseudo_df = pd.DataFrame({
    'query_id':list(hyde_pseudo.keys()),
    'query_text':queries_df['query_text'].tolist(),
    'pseudo_answer':list(hyde_pseudo.values())
})

hyde_pseudo_df.head()

,query_id,query_text,pseudo_answer
0,Q1,제주도 올레길 트레킹 코스 추천,"제주도 올레길은 총 26개 코스로 구성되어 있으며, 각 코스마다 독특한 자연경관과 문화유산을 경험할 수 있습니다. 특히 올레길 7코스는 한라산과 바다를 동시에 감상할 수 있어 인기 있으며, 10코스는 서귀포의 아름다운 해안선을 따라 걷기에 좋습니다. 트레킹 난이도와 소요 시간은 코스마다 다르므로, 자신의 체력과 일정에 맞는 코스를 선택하는 것이 중요합니다. 제주도 올레길 공식 홈페이지나 여행 후기 사이트에서 상세 정보를 확인해보는 것을 추천합니다."
1,Q2,전주 비빔밥 vs 진주 비빔밥 재료 및 맛 차이,"전주 비빔밥과 진주 비빔밥은 재료와 맛에서 차이가 있습니다. 전주 비빔밥은 다양한 나물과 고기, 계란 지단, 고추장 양념이 조화롭게 어우러져 고소하고 풍부한 맛이 특징입니다. 반면 진주 비빔밥은 생선회와 육회, 다양한 해산물 재료가 포함되어 신선하고 담백한 맛을 냅니다. 두 지역 비빔밥 모두 지역 특산물과 조리법에 따라 맛과 재료 구성이 다르므로 자세한 비교를 위해 각 지역의 전통 조리법을 참고하는 것이 좋습니다."
2,Q3,걸스데이 대표 히트곡 목록 알려줘,"걸스데이 대표 히트곡으로는 ""기대해"", ""Something"", ""Darling"", ""Ring My Bell"" 등이 자주 언급됩니다. 이 곡들은 걸스데이의 음악적 변화를 보여주며 대중적인 인기를 얻은 노래들입니다. 특히 ""Something""은 섹시 콘셉트로 큰 주목을 받았고, ""기대해""는 초기 히트곡으로 팬들에게 사랑받았습니다. 걸스데이 히트곡 목록을 더 자세히 알고 싶다면 음악 차트 기록이나 공식 팬사이트를 참고하는 것이 좋습니다."
3,Q4,훈민정음 창제 배경과 세종대왕의 의의,"훈민정음은 15세기 조선 세종대왕이 백성들의 문자 사용을 돕기 위해 창제한 한글의 원래 이름입니다. 당시 한자는 배우기 어려워 일반 백성들이 의사소통에 어려움을 겪었고, 세종대왕은 이를 해결하고자 쉬운 문자 체계를 만들고자 했습니다. 훈민정음 창제는 국민의 문자 생활을 혁신하고 문화 발전에 큰 기여를 한 중요한 역사적 사건으로 평가됩니다. 세종대왕은 이를 통해 백성 중심의 통치와 교육 확산에 큰 의의를 가진 인물로 기억됩니다."
4,Q5,이순신 장군이 명량 해전에서 사용한 전술은 무엇인가?,"이순신 장군이 명량 해전에서 사용한 전술은 좁은 해협 지형을 활용한 방어적 기동전술로 알려져 있습니다. 그는 조류가 빠른 명량 해협의 지리적 특성을 이용해 적의 대규모 함대를 효과적으로 분산시키고, 조선 수군의 판옥선과 거북선을 중심으로 집중 공격을 펼쳤습니다. 이 전술은 적의 수적 우위를 극복하는 데 중요한 역할을 했으며, 해상 전투에서 지형과 조류를 전략적으로 활용한 사례로 자주 언급됩니다. 자세한 전술 분석은 역사 기록과 군사 전략 자료에서 확인할 수 있습니다."


## 성능 평가 함수

In [12]:
import numpy as np

def parse_relevant(relevant_str):
    """다중 정답 및 등급을 처리하기 위한 헬퍼 함수"""
    pairs = relevant_str.split(";")
    rel_dict = {}
    for pair in pairs:
        doc_id, grade = pair.split("=")
        rel_dict[doc_id] = grade
    return rel_dict 

def compute_metrics(predicted, relevant_dict, k=5):
    relevant_docs = set(relevant_dict.keys())
    top_k = predicted[:k]
    hits = sum(1 for doc in top_k if doc in relevant_docs)
    precision = hits / k
    total_relevant = len(relevant_docs)
    recall = hits / total_relevant if total_relevant > 0 else 0 
    rr = 0
    for idx, doc in enumerate(top_k):
        if doc in relevant_docs:
            rr = 1 / (idx + 1)
            break
    num_correct = 0
    precision_sum = 0
    for i, doc in enumerate(top_k):
        if doc in relevant_docs:
            num_correct += 1
            precision_sum += num_correct / (i + 1)
    denominator = min(total_relevant, k)
    ap = precision_sum / denominator if denominator > 0 else 0
    return precision, recall, rr, ap

def evaluate_all(method_results, queries_df, k=5):
    prec_list, rec_list, rr_list, ap_list = [], [], [], []
    for idx, row in queries_df.iterrows():
        qid = row['query_id']
        relevant_dict = parse_relevant(row['relevant_doc_ids'])
        predicted = method_results[qid]
        p, r, rr, ap = compute_metrics(predicted, relevant_dict, k)
        prec_list.append(p)
        rec_list.append(r)
        rr_list.append(rr)
        ap_list.append(ap)
    return {
        'Precision@k' : np.mean(prec_list),
        'Recall@k' : np.mean(rec_list),
        'MRR' : np.mean(rr_list),
        'MAP' : np.mean(ap_list),
    }

In [ ]:
# BM25/Dense/HyDE 검색 결과를 같은 형식으로 만든다.
bm25_results = {}
for id,row in queries_df.iterrows():
    qid = row['query_id']
    query_text = row['query_text']
    bm25_results[qid] = bm25_search(query_text,top_k=5)

dense_results = {}
for id,row in queries_df.iterrows():
    qid = row['query_id']
    query_text = row['query_text']
    dense_results[qid] = dense_search(query_text,top_k=5)

hyde_results = {}
for id,row in hyde_pseudo_df.iterrows():
    qid = row['query_id']
    query_text = row['pseudo_answer']
    hyde_results[qid] = dense_search(query_text,top_k=5)


In [ ]:
bm25_metrics =  evaluate_all(bm25_results,queries_df)
dense_metrics =  evaluate_all(dense_results,queries_df)
hyde_metrics = evaluate_all(hyde_results,queries_df)

In [19]:
metrics_df = pd.DataFrame({
    'Metric' : ["Precision@k","Recall@k","MRR","MAP"],
    'BM25' : [bm25_metrics["Precision@k"],bm25_metrics["Recall@k"],bm25_metrics["MRR"],bm25_metrics["MAP"]],
    'Dense' : [dense_metrics["Precision@k"],dense_metrics["Recall@k"],dense_metrics["MRR"],dense_metrics["MAP"]],
    'HyDE' : [hyde_metrics["Precision@k"],hyde_metrics["Recall@k"],hyde_metrics["MRR"],hyde_metrics["MAP"]],
})
metrics_df

,Metric,BM25,Dense,HyDE
0,Precision@k,0.246667,0.233333,0.246667
1,Recall@k,1.000000,0.975000,1.000000
2,MRR,0.983333,1.000000,1.000000
3,MAP,0.977778,0.975000,0.994444
